In [ ]:
# ============================================================
# STEP 1 : Import Libraries
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

PROJECT_DIR = PROJECT_ROOT
OUTPUT_DIR = OUTPUT_ROOT / "cwc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# STEP 3 : Load Dataset
# ============================================================

master = pd.read_csv(
    master_file,
    parse_dates=["Data Acquisition Time"]
)

print(master.shape)

master.head()

In [ ]:
# ============================================================
# STEP 4 : Dataset Overview
# ============================================================

print("="*60)
print("DATASET SUMMARY")
print("="*60)

print("Rows :", len(master))
print("Columns :", len(master.columns))

print("Stations :", master["Station_UID"].nunique())

print("Date Range :")
print(master["Data Acquisition Time"].min())
print(master["Data Acquisition Time"].max())

In [ ]:
# ============================================================
# STEP 5 : Data Types
# ============================================================

dtype_summary = pd.DataFrame({

    "Column": master.columns,

    "Datatype": master.dtypes.astype(str),

    "Missing": master.isna().sum(),

    "Missing_%": (
        master.isna().mean()*100
    ).round(2)

})

display(dtype_summary)

In [ ]:
# ============================================================
# STEP 6 : Metadata Columns
# ============================================================

metadata_cols = [

    "Station_UID",
    "Station",
    "Latitude",
    "Longitude",
    "State",
    "District",
    "Tehsil",
    "Basin",
    "Data Acquisition Time",
    "Source_File"

]

In [ ]:
# ============================================================
# STEP 7 : Biological Columns
# ============================================================

bio_cols = [

    c
    for c in master.columns

    if (
        "coliform" in c.lower()
        or
        "streptococci" in c.lower()
    )

]

print(bio_cols)

In [ ]:
# ============================================================
# STEP 8 : Define Target
# ============================================================

TARGET = "Total Coliform (MPN/100mL)"

print("Target :", TARGET)

In [ ]:
# ============================================================
# STEP 9 : Remove Non-ML Columns
# ============================================================

drop_cols = [
    "Station_UID",
    "Station",
    "State",
    "District",
    "Tehsil",
    "Basin",
    "Source_File"
]

ml_df = master.drop(columns=drop_cols, errors="ignore")

print(ml_df.shape)

In [ ]:
# ============================================================
# STEP 10 : Split Features and Target
# ============================================================

X = ml_df.drop(columns=[TARGET])

y = ml_df[TARGET]

print("Features :", X.shape)
print("Target   :", y.shape)

In [ ]:
# ============================================================
# STEP 11 : Missing Value Report
# ============================================================

missing_report = pd.DataFrame({

    "Column": X.columns,
    "Datatype": X.dtypes.astype(str),
    "Missing": X.isna().sum(),
    "Missing_%": (X.isna().mean() * 100).round(2),
    "Unique": X.nunique()

})

missing_report = (
    missing_report
    .sort_values("Missing_%", ascending=False)
    .reset_index(drop=True)
)

display(missing_report)

print("Columns with Missing Values :",
      (missing_report["Missing"] > 0).sum())

In [ ]:
missing_report.to_csv(
    OUTPUT_DIR / "Missing_Value_Report.csv",
    index=False
)

print("Report Saved")

In [ ]:
# ============================================================
# STEP 13 : Remove High Missing Columns
# ============================================================

THRESHOLD = 70

drop_missing = (
    missing_report.loc[
        missing_report["Missing_%"] >= THRESHOLD,
        "Column"
    ]
    .tolist()
)

print("Columns to Drop :", len(drop_missing))
print(drop_missing)

ml_df = ml_df.drop(columns=drop_missing)

print(ml_df.shape)

In [ ]:
# ============================================================
# STEP 14 : Date Features
# ============================================================

ml_df["Year"] = master["Data Acquisition Time"].dt.year
ml_df["Month"] = master["Data Acquisition Time"].dt.month
ml_df["Day"] = master["Data Acquisition Time"].dt.day

ml_df["Quarter"] = master["Data Acquisition Time"].dt.quarter

In [ ]:
def season(month):

    if month in [12, 1, 2]:
        return "Winter"

    elif month in [3, 4, 5]:
        return "Summer"

    elif month in [6, 7, 8, 9]:
        return "Monsoon"

    return "Post-Monsoon"


ml_df["Season"] = (
    ml_df["Month"]
    .apply(season)
)

In [ ]:
ml_df.drop(
    columns=["Data Acquisition Time"],
    inplace=True
)

In [ ]:
# ============================================================
# STEP 16 : Feature Types
# ============================================================

feature_cols = [c for c in ml_df.columns if c != TARGET]

numeric_cols = ml_df[feature_cols].select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = ml_df[feature_cols].select_dtypes(
    include=["object"]
).columns.tolist()

print("="*60)
print("Numeric Features     :", len(numeric_cols))
print("Categorical Features :", len(categorical_cols))
print("="*60)

print("\nCategorical Columns:")
print(categorical_cols)

In [ ]:
# ============================================================
# STEP 17 : Missing Value Treatment
# ============================================================

for col in numeric_cols:
    ml_df[col] = ml_df[col].fillna(
        ml_df[col].median()
    )

for col in categorical_cols:
    ml_df[col] = ml_df[col].fillna(
        ml_df[col].mode()[0]
    )

In [ ]:
# ============================================================
# STEP 18 : Verify Missing Values
# ============================================================

missing = ml_df.isna().sum().sum()

print("Remaining Missing Values :", missing)

In [ ]:
# ============================================================
# STEP 19 : One-Hot Encoding
# ============================================================

ml_ready = pd.get_dummies(
    ml_df,
    columns=categorical_cols,
    drop_first=True
)

print(ml_ready.shape)

In [ ]:
# ============================================================
# STEP 20 : Final Validation
# ============================================================

print("="*60)

print("Rows :", len(ml_ready))
print("Columns :", len(ml_ready.columns))

print("Missing :", ml_ready.isna().sum().sum())

print("="*60)

In [ ]:
missing_after = (
    ml_ready.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_after = missing_after[missing_after > 0]

display(missing_after)

In [ ]:
print(master[TARGET].isna().sum())

In [ ]:
for col in numeric_cols:
    if ml_df[col].isna().sum() > 0:
        print(col, ml_df[col].isna().sum())

In [ ]:
for col in categorical_cols:
    if ml_df[col].isna().sum() > 0:
        print(col, ml_df[col].isna().sum())

In [ ]:
print(categorical_cols)

for col in categorical_cols:
    print(col, ml_df[col].nunique())

In [ ]:
ml_df = ml_df.dropna(subset=[TARGET]).reset_index(drop=True)

print(ml_df.shape)

In [ ]:
drop_extra = [

    "Agency",
    "Agency_x",
    "Agency_y",

    "Block",
    "Block_x",
    "Block_y",

    "Village",
    "Village_x",
    "Village_y",

    "River",
    "River_x",
    "River_y",

    "Tribuary",
    "Tributary_x",
    "Tributary_y",

    "Subtributary",
    "Subtributary_x",
    "Subtributary_y",

    "SubSubtributary",
    "SubSubtributary_x",
    "SubSubtributary_y",

    "Local River",
    "Local River_x",
    "Local River_y",

    "Merge_Key",
    "Merge_Key_x",
    "Merge_Key_y",

    "Period",
    "Period_x",
    "Period_y",
]

In [ ]:
ml_df.drop(
    columns=drop_extra,
    errors="ignore",
    inplace=True
)

In [ ]:
feature_cols = [c for c in ml_df.columns if c != TARGET]

numeric_cols = ml_df[feature_cols].select_dtypes(include="number").columns.tolist()

categorical_cols = ml_df[feature_cols].select_dtypes(include="object").columns.tolist()

print(categorical_cols)

In [ ]:
print(ml_df["Tributary"].value_counts(dropna=False))
print(ml_df["Tributary"].nunique())

In [ ]:
# ============================================================
# STEP 32 : Fill Missing Values
# ============================================================

for col in numeric_cols:
    ml_df[col] = ml_df[col].fillna(ml_df[col].median())

for col in categorical_cols:
    ml_df[col] = ml_df[col].fillna(ml_df[col].mode()[0])

print("Remaining Missing :", ml_df.isna().sum().sum())

In [ ]:
# ============================================================
# STEP 33 : One-Hot Encoding
# ============================================================

ml_ready = pd.get_dummies(
    ml_df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print(ml_ready.shape)
print("Missing :", ml_ready.isna().sum().sum())

In [ ]:
# ============================================================
# STEP 34 : Save ML Ready Dataset
# ============================================================

ml_file = OUTPUT_DIR / "ML_Ready_Dataset.csv"

ml_ready.to_csv(
    ml_file,
    index=False
)

print("Saved Successfully")
print(ml_file)